In [84]:
import pandas as pd


# CUSTOMER


In [85]:
lien = "https://raw.githubusercontent.com/HanitsoaAyan/ETL_pipeline_data/refs/heads/main/customers.csv"


In [86]:
df_customer = pd.read_csv(lien)


In [87]:
df_customer.head(10)

,customer_id,name,email,signup_date,country,age
0,1,Alice Dupont,alice@example.com,2023-01-15,France,29.0
1,2,Bob Martin,bob@example.com,2023-02-20,France,NaN
2,3,Chloé Bernard,chloe@example.com,2023-03-05,Belgium,34.0
3,4,David Leroy,NaN,2023-04-10,France,41.0
4,5,Alice Dupont,alice@example.com,2023-01-15,France,29.0
5,6,Eva Petit,eva@example.com,2023-05-22,Switzerland,120.0
6,7,Farid Nasri,farid@example.com,NaN,Morocco,27.0
7,8,Gabrielle Roy,gabrielle@example.com,2023-06-30,France,31.0


In [88]:
df_customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   customer_id  8 non-null      int64  
 1   name         8 non-null      object 
 2   email        7 non-null      object 
 3   signup_date  7 non-null      object 
 4   country      8 non-null      object 
 5   age          7 non-null      float64
dtypes: float64(1), int64(1), object(4)
memory usage: 516.0+ bytes


In [89]:
def quality_check_customers(df):
    rapport = {
        "lignes_totales": len(df),
        "valeurs_manquantes_par_colonne": df.isna().sum().to_dict(),
        "verification_doublons_par_mail": int(df.duplicated(subset=["email"]).sum()),
        "age_invalide":int(((df["age"] < 0) | (df["age"] > 110)).sum()),
        "emails_mal_formes": int((~df["email"].dropna().astype(str).str.contains("@")).sum())
    }
    return rapport

quality_check_customers(df_customer)

{'lignes_totales': 8,
 'valeurs_manquantes_par_colonne': {'customer_id': 0,
  'name': 0,
  'email': 1,
  'signup_date': 1,
  'country': 0,
  'age': 1},
 'verification_doublons_par_mail': 1,
 'age_invalide': 1,
 'emails_mal_formes': 0}

In [90]:
def transform_customers(df):
    df = df.copy()
    df["name"] = df["name"].str.strip()
    df = df.drop_duplicates(subset=["email"], keep="first")
    df["age"] = df["age"].where(df["age"].between(0, 110))
    return df
df_clean = transform_customers(df_customer)
df_clean


,customer_id,name,email,signup_date,country,age
0,1,Alice Dupont,alice@example.com,2023-01-15,France,29.0
1,2,Bob Martin,bob@example.com,2023-02-20,France,NaN
2,3,Chloé Bernard,chloe@example.com,2023-03-05,Belgium,34.0
3,4,David Leroy,NaN,2023-04-10,France,41.0
5,6,Eva Petit,eva@example.com,2023-05-22,Switzerland,NaN
6,7,Farid Nasri,farid@example.com,NaN,Morocco,27.0
7,8,Gabrielle Roy,gabrielle@example.com,2023-06-30,France,31.0


In [91]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7 entries, 0 to 7
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   customer_id  7 non-null      int64  
 1   name         7 non-null      object 
 2   email        6 non-null      object 
 3   signup_date  6 non-null      object 
 4   country      7 non-null      object 
 5   age          5 non-null      float64
dtypes: float64(1), int64(1), object(4)
memory usage: 692.0+ bytes


# Order

In [92]:
df_orders= pd.read_json("https://raw.githubusercontent.com/HanitsoaAyan/ETL_pipeline_data/refs/heads/main/orders.json")
df_orders.head()

,order_id,customer_id,amount,status,order_date
0,101,1,59.9,completed,2023-01-20
1,102,2,120.0,pending,2023-02-25
2,103,3,15.5,completed,2023-03-10
3,104,1,NaN,cancelled,2023-04-01
4,105,6,9999.0,completed,2023-05-25


In [93]:
df_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     6 non-null      int64  
 1   customer_id  6 non-null      int64  
 2   amount       5 non-null      float64
 3   status       6 non-null      object 
 4   order_date   6 non-null      object 
dtypes: float64(1), int64(2), object(2)
memory usage: 372.0+ bytes


In [94]:
df_orders.describe()

,order_id,customer_id,amount
count,6.000000,6.000000,5.000000
mean,103.500000,3.500000,2047.920000
std,1.870829,2.880972,4444.951931
min,101.000000,1.000000,15.500000
25%,102.250000,1.250000,45.200000
50%,103.500000,2.500000,59.900000
75%,104.750000,5.250000,120.000000
max,106.000000,8.000000,9999.000000


In [95]:
def quality_check_orders(df):
    rapport = {
        "lignes_totales": len(df),
        "valeurs_manquantes_par_colonne": df.isna().sum().to_dict(),
        "verification_doublons": int(df.duplicated().sum()),
        "montant_manquant_statut_different_annule": int((df["amount"].isna() & (df["status"] != "cancelled")).sum()),
        "montant_invalide": int((df["amount"] < 0).sum()),
        "montant_abberant": int((df["amount"] > 5000).sum())
    }
    return rapport

quality_check_orders(df_orders)

{'lignes_totales': 6,
 'valeurs_manquantes_par_colonne': {'order_id': 0,
  'customer_id': 0,
  'amount': 1,
  'status': 0,
  'order_date': 0},
 'verification_doublons': 0,
 'montant_manquant_statut_different_annule': 0,
 'montant_invalide': 0,
 'montant_abberant': 1}

In [100]:
def transform_orders(df):
    df = df.copy()
    masque_annule = (df["amount"].isna()) & (df["status"] == "cancelled")
    df.loc[masque_annule, "amount"] = 0
    return df

# --- Appel de la fonction (hors de la fonction, tout à gauche) ---
df_clean = transform_orders(df_orders)
df_clean

,order_id,customer_id,amount,status,order_date
0,101,1,59.9,completed,2023-01-20
1,102,2,120.0,pending,2023-02-25
2,103,3,15.5,completed,2023-03-10
3,104,1,0.0,cancelled,2023-04-01
4,105,6,9999.0,completed,2023-05-25
5,106,8,45.2,completed,2023-07-02


In [ ]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     6 non-null      int64  
 1   customer_id  6 non-null      int64  
 2   amount       6 non-null      float64
 3   status       6 non-null      object 
 4   order_date   6 non-null      object 
dtypes: float64(1), int64(2), object(2)
memory usage: 372.0+ bytes


#Products

In [ ]:
df_products = pd.read_csv("https://raw.githubusercontent.com/HanitsoaAyan/ETL_pipeline_data/refs/heads/main/products.csv")

In [ ]:
df_products.head()

,product_id,name,category,price,stock
0,1,Clavier mécanique,Informatique,79.99,120
1,2,Souris sans fil,Informatique,29.90,300
2,3,Casque audio,Audio,59.50,85
3,4,Chaise de bureau,Mobilier,149.00,40
4,5,Lampe de bureau,Mobilier,24.99,200


In [58]:
df_products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   product_id  8 non-null      int64  
 1   name        8 non-null      object 
 2   category    8 non-null      object 
 3   price       8 non-null      float64
 4   stock       8 non-null      int64  
dtypes: float64(1), int64(2), object(2)
memory usage: 452.0+ bytes


In [59]:
df_products.describe()

,product_id,price,stock
count,8.00000,8.000000,8.000000
mean,4.50000,54.783750,181.875000
std,2.44949,43.695204,153.388803
min,1.00000,9.990000,40.000000
25%,2.75000,28.672500,78.750000
50%,4.50000,42.450000,135.000000
75%,6.25000,64.622500,225.000000
max,8.00000,149.000000,500.000000


In [65]:
def quality_check_products(df):
  rapport = {
        "lignes_totales": len(df),
        "valeurs_manquantes_par_colonne": df.isna().sum().to_dict(),
        "verification_doublons": int(df.duplicated().sum()),
        "prix_invalide": int(((df["price"] < 0) | (df["price"] == 0)).sum()),
        "stock_invalide" : int(((df["stock"]) < 0).sum()),
       "prix_abberants": int((df["price"] > 5000).sum()),
        "type_prix" : bool(pd.api.types.is_numeric_dtype(df["price"])),
        "type_stock" : bool(pd.api.types.is_numeric_dtype(df["stock"]))
  }
  return rapport

quality_check_products(df_products)

{'lignes_totales': 8,
 'valeurs_manquantes_par_colonne': {'product_id': 0,
  'name': 0,
  'category': 0,
  'price': 0,
  'stock': 0},
 'verification_doublons': 0,
 'prix_invalide': 0,
 'stock_invalide': 0,
 'prix_abberants': 0,
 'type_prix': True,
 'type_stock': True}

In [101]:
def transform_products(df):
    df = df.copy()
    # aucune anomalie détectée actuellement — fonction prête si le schéma évolue
    return df

# Reviews

In [ ]:
df_reviews= pd.read_json("https://raw.githubusercontent.com/HanitsoaAyan/ETL_pipeline_data/refs/heads/main/reviews.json")

In [ ]:
df_reviews.head()

,review_id,product_id,rating,comment,review_date
0,1,1,5,"Excellent clavier, très réactif",2024-01-10
1,2,2,4,"Bonne souris, autonomie correcte",2024-01-15
2,3,1,3,Bruyant mais efficace,2024-02-02
3,4,3,5,Son excellent pour le prix,2024-02-20
4,5,4,2,Confort décevant,2024-03-01


In [66]:
df_reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   review_id    5 non-null      int64 
 1   product_id   5 non-null      int64 
 2   rating       5 non-null      int64 
 3   comment      5 non-null      object
 4   review_date  5 non-null      object
dtypes: int64(3), object(2)
memory usage: 332.0+ bytes


In [67]:
df_reviews.describe()

,review_id,product_id,rating
count,5.000000,5.00000,5.00000
mean,3.000000,2.20000,3.80000
std,1.581139,1.30384,1.30384
min,1.000000,1.00000,2.00000
25%,2.000000,1.00000,3.00000
50%,3.000000,2.00000,4.00000
75%,4.000000,3.00000,5.00000
max,5.000000,4.00000,5.00000


In [76]:
def quality_check_reviews(df_reviews, df_products):
    rapport = {
        "lignes_totales": len(df_reviews),
        "valeurs_manquantes_par_colonne": df_reviews.isna().sum().to_dict(),
        "verification_doublons": int(df_reviews.duplicated().sum()),
        "product_id_inexistants": int((~df_reviews["product_id"].isin(df_products["product_id"])).sum()),
        "notes_toutes_valides": bool(df_reviews["rating"].between(1, 5).all()),
        "espaces_superflus": int((df_reviews["comment"].astype(str) != df_reviews["comment"].astype(str).str.strip()).sum()),
        "commentaires_trop_courts": int((df_reviews["comment"].astype(str).str.strip().str.len() < 2).sum())
    }
    return rapport


resultat = quality_check_reviews(df_reviews, df_products)
print(resultat)

{'lignes_totales': 5, 'valeurs_manquantes_par_colonne': {'review_id': 0, 'product_id': 0, 'rating': 0, 'comment': 0, 'review_date': 0}, 'verification_doublons': 0, 'product_id_inexistants': 0, 'notes_toutes_valides': True, 'espaces_superflus': 0, 'commentaires_trop_courts': 0}


In [102]:
def transform_reviews(df):
    df = df.copy()
    df["comment"] = df["comment"].str.strip()
    return df

# Relation

In [ ]:
df_relation = pd.read_json("https://raw.githubusercontent.com/HanitsoaAyan/ETL_pipeline_data/refs/heads/main/raw/run-relations/relations.json")

In [ ]:
df_relation.head()

,customer_id,name,order_id,amount,status
0,1,Alice Dupont,101,59.9,completed
1,2,Bob Martin,102,120.0,pending


In [78]:
def quality_check_relation(df, minimum_attendu=4):
    rapport = {
        "lignes_totales": len(df),
        "valeurs_manquantes_par_colonne": df.isna().sum().to_dict(),
        "volume_suspect": len(df) < minimum_attendu,
    }
    return rapport

quality_check_relation(df_relation)

{'lignes_totales': 2,
 'valeurs_manquantes_par_colonne': {'customer_id': 0,
  'name': 0,
  'order_id': 0,
  'amount': 0,
  'status': 0},
 'volume_suspect': True}

# Utilisateur

In [ ]:
df_utilisateur= pd.read_json("https://raw.githubusercontent.com/HanitsoaAyan/ETL_pipeline_data/refs/heads/main/raw/run-utilisateurs/utilisateurs.json")
df_utilisateur.head()

,numero,credit
0,22,20
1,24,860


In [77]:
def quality_check_utilisateur(df, minimum_attendu=5):
    rapport = {
        "lignes_totales": len(df),
        "valeurs_manquantes_par_colonne": df.isna().sum().to_dict(),
        "credit_negatif": int((df["credit"] < 0).sum()),
        "volume_suspect": len(df) < minimum_attendu,
    }
    return rapport

quality_check_utilisateur(df_utilisateur)

{'lignes_totales': 2,
 'valeurs_manquantes_par_colonne': {'numero': 0, 'credit': 0},
 'credit_negatif': 0,
 'volume_suspect': True}

In [81]:
def transform_utilisateur(df):
    df = df.copy()
    df["credit_negatif"] = df["credit"] < 0
    return df
